# Reflection and Blogpost Writing

## Setup

In [1]:

llm_config = {"model": "llama3.2","api_type":"ollama"}


## The task!

In [2]:
task = '''
        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

Patching name='__init__', member=<function LLMLingua.__init__ at 0x000001A62BFED260>, patched=<function function.__call__ at 0x000001A62BFED1C0>
Patching name='compress_text', member=<function LLMLingua.compress_text at 0x000001A62BFED300>, patched=<function function.__call__ at 0x000001A62BFED580>


In [4]:
reply = writer.generate_reply(messages=[{"content": task ,"role": "user"}])

In [5]:
print(reply)

{'content': "**Unlocking Human Potential: The Power of Hope AI**\n\nImagine having a personal assistant that not only automates tasks but also understands your emotions and provides support when you need it most. Welcome to Hope AI, an innovative technology that harnesses the power of artificial intelligence to help humans thrive. This cutting-edge system uses machine learning algorithms to analyze user data, detect emotional cues, and offer personalized guidance and encouragement. With Hope AI, you can overcome obstacles, build resilience, and achieve your goals with confidence. By combining human ingenuity with technological advancements, we're on the cusp of a revolutionary era that redefines what it means to be human.", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Unlocking Human Potential with Hope AI**

Hope AI, developed by Stanford University researchers, has sparked excitement in the tech community. This innovative AI system claims to help individuals overcome debilitating mental health conditions, such as depression and anxiety. By leveraging machine learning algorithms and natural language processing, Hope AI aims to provide personalized support and guidance. Preliminary results suggest impressive gains in users' symptoms and overall well-being. While more research is needed, Hope AI represents a promising breakthrough in the pursuit of mental wellness. Could this groundbreaking tech be the key to unlocking human potential?

-----------------------------------------------------------------

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Unlocking Human Potential with Hope AI**

Hope AI, developed by Stanford University researchers, has sparked excitement in the tech community. This innovative AI system claims to help individuals overcome debilitating mental health conditions, such as depression and anxiety. By leveraging machine learning algorithms and natural language processing, Hope AI aims to provide personalized support and guidance. Preliminary results suggest impressive gains in users' symptoms and overall well-being. While more research is needed, Hope AI represents a promising breakthrough in the pursuit of mental wellness. Could this groundbreaking tech be the key to unlocking human potential?

-----------------------------------------------------------------

## Get the summary

In [15]:
print(res.summary)

**Unlocking Human Potential with Hope AI**

Hope AI, developed by researchers at Stanford University, is an innovative technology aimed at supporting mental wellness. This machine learning-based system leverages natural language processing to provide personalized guidance and support for individuals struggling with anxiety and depression. While showing promise, it's essential to understand the limitations of Hope AI. Its capabilities are not a replacement for medical supervision or professional treatment. Users should consult with a qualified healthcare professional before using the system.

**Disclaimer:** Hope AI is not intended to diagnose, treat, or cure mental health conditions. Its capabilities are limited, and users should use it in conjunction with expert guidance.

By acknowledging the limitations of Hope AI, we can promote responsible innovation and empower individuals to make informed decisions about their mental health.
